# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

We train and evaluate three competitive model families:
1. **Logistic Regression (L2 Regularized):** Serves as a calibrated linear model; establishes how much performance is explained by additive linear combinations of signals.
2. **Decision Tree (Max Depth 6):** Captures single-split non-linear threshold rules without complex ensembles.
3. **Random Forest (100 Estimators):** Combines bagged decision trees with feature sub-sampling to model high-order interactions between ranking, velocity, and engagement while preventing overfitting.

In [1]:
import json
from pathlib import Path
import pandas as pd

results_path = Path("../../outputs/model_results.json")
with open(results_path) as f:
    results = json.load(f)

print(f"Loaded evaluation results from {results_path}")
print(f"Best model selected: {results['best_model']['name']} via {results['best_model']['selection_metric']}")

Loaded evaluation results from ../../outputs/model_results.json
Best model selected: random_forest via precision_at_50


## 2. Split design

- **Strategy:** `client_holdout` (Group split by client).
- **Rationale:** Standard random row splitting causes severe data leakage because pages from the same client share domain authority, technical infrastructure, and publication velocity. By holding out 4 entire clients (2,325 rows) and training on 28 clients (27,675 rows), we simulate real-world generalization to new domains.
- **Split Sizes:** Train = 27,675 rows (92.25%), Test = 2,325 rows (7.75%).

In [2]:
print(f"Training split rows: {results['train_rows']:,}")
print(f"Holdout test split rows: {results['test_rows']:,}")
print(f"Target positive rate in full dataset: {results['target_positive_rate']:.4f}")

Training split rows: 27,675
Holdout test split rows: 2,325
Target positive rate in full dataset: 0.5421


## 3. Train + compare vs my baseline

All models and the baseline heuristic rule are evaluated on the exact same held-out client test set:

| Model | Precision@20 | Precision@50 | Precision@100 | ROC-AUC | PR-AUC |
|---|---:|---:|---:|---:|---:|
| **Baseline Rules** | 0.150 | 0.240 | 0.360 | 0.627 | 0.468 |
| **Logistic Regression** | 0.350 | 0.400 | 0.440 | 0.700 | 0.522 |
| **Decision Tree** | 0.450 | 0.580 | 0.620 | 0.742 | 0.575 |
| **Random Forest** | **0.700** | **0.680** | **0.700** | **0.747** | **0.610** |

**Key Takeaway:** Random Forest achieves **Precision@50 = 0.680**, representing a **2.83× lift** over the heuristic baseline (0.240) on unseen client data.

In [3]:
models = results['models']
baseline = results['baseline']

comparison_data = [
    {
        'Model': 'Baseline Rules',
        'Precision@20': baseline['baseline_precision_at_20'],
        'Precision@50': baseline['baseline_precision_at_50'],
        'Precision@100': baseline['baseline_precision_at_100'],
        'ROC-AUC': baseline['baseline_roc_auc'],
        'PR-AUC': baseline['baseline_average_precision']
    }
]

for name, m in models.items():
    comparison_data.append({
        'Model': name.replace('_', ' ').title(),
        'Precision@20': m['precision_at_20'],
        'Precision@50': m['precision_at_50'],
        'Precision@100': m['precision_at_100'],
        'ROC-AUC': m['roc_auc'],
        'PR-AUC': m['average_precision']
    })

comp_df = pd.DataFrame(comparison_data)
print(comp_df.to_string(index=False))

              Model  Precision@20  Precision@50  Precision@100  ROC-AUC   PR-AUC
     Baseline Rules          0.15          0.24           0.36 0.626892 0.467607
      Decision Tree          0.45          0.58           0.62 0.741520 0.575319
Logistic Regression          0.35          0.40           0.44 0.700291 0.521542
      Random Forest          0.70          0.68           0.70 0.747372 0.610058


## 4. Errors and interpretation

### Feature Importances:
The top 5 predictive features identified by Random Forest:
1. `days_with_impressions` (16.06%): Consistency of search presence over time.
2. `log_impressions_90d` (12.85%): Total search demand exposure.
3. `avg_position` (10.84%): Current Google rank position.
4. `content_age_days` (9.50%): Time elapsed since creation.
5. `word_count` (4.12%): Content depth and format length.

### Error Analysis:
- **False Positives:** Pages with high historical impressions and declining positions that stabilized naturally without editorial change.
- **False Negatives:** Thin, low-impression articles that collapsed due to off-page factor changes not captured in search logs.

In [4]:
top_features = pd.DataFrame(results['best_model']['feature_importance_top'][:10])
print("Top 10 Feature Importances (Random Forest):")
print(top_features.to_string(index=False))

Top 10 Feature Importances (Random Forest):
              feature  importance
days_with_impressions    0.160612
  log_impressions_90d    0.128491
         avg_position    0.108389
     content_age_days    0.094997
           word_count    0.041163
           char_count    0.039843
                  ctr    0.033164
       log_clicks_90d    0.032499
          scroll_rate    0.030914
   days_with_sessions    0.029878


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.